In [17]:
from sympy import *
from IPython.display import *

def geo(m):
    """ Вывод матрицы в формате геолина """
    if m.shape[0] == 1:
        return f'[{', '.join([str(round(m[0, i], 10)) for i in range(m.shape[1])])}]'
    return f'[{'; '.join([', '.join([str(round(m[j, i], 10)) for i in range(m.shape[1])]) for j in range(m.shape[0])])}]'    

def geolin_matrix(s: str, n: int) -> Matrix:
    """ Ввод матрицы по столбцам """
    s = s.replace('\u2212', '-').replace('\u200b', '').split()
    d = MutableDenseMatrix.zeros(n, len(s) // n)
    for i in range(len(s)):
        d[i % n, i // n] = int(s[i])
    return d

def geolin_vector(s: str) -> Matrix:
    """ Ввод вектора """
    s = s.replace('\u2212', '-').replace('\u200b', '').split()
    return Matrix([int(x) for x in s])

In [18]:
def lagrange(Q: Matrix) -> tuple[Matrix, Matrix]:
    display(Latex("Исходная квадратичная форма:"), Q)
    display(Latex("Метод Лагранжа"))
    n = Q.rows
    Td = Matrix.eye(n)  # Матрица перехода (единичная)
    
    for k in range(n):
        # Проверка нулевого диагонального элемента
        if Q[k, k] == 0:
            # Поиск ненулевого элемента для перестановки
            for j in range(k + 1, n):
                if Q[k, j] != 0:
                    # Перестановка строк и столбцов
                    Q = Q.swap_rows(k, j)
                    Q = Q.swap_cols(k, j)
                    Td = Td.swap_cols(k, j)
                    display(Latex(f"Меняем местами $v_{{{k}}}$ и $v_{{{j}}}$"))
                    break
            else:
                display(Latex(f"Пропускаем $v_{{{k}}}$ (весь столбец нулевой)"))
                continue
        
        # Формирование замены для текущего шага
        out = f"\\tilde{{v_{{{k}}}}} = v_{{{k}}}"
        S_k = Matrix.eye(n)  # Матрица элементарной замены
        for j in range(k + 1, n):
            coef = - 2 * Q[k, j] / Q[k, k]  # Отрицательный коэффициент!
            S_k[k, j] = coef
            # Форматирование вывода
            if coef < 0:
                out += f" - {-coef:.4f} v_{{{j}}}"
            else:
                out += f" + {coef:.4f} v_{{{j}}}"
        
        display(Math(out))
        
        # Применение замены к матрицам
        Q = S_k.T * Q * S_k
        Td = Td * S_k
    
    display(Latex("Матрица перехода (столбцы - координаты новых векторов):"), Td)
    display(Latex("Диагонализированная квадратичная форма:"), Q)
    return Q, Td

In [19]:
Q = Matrix([
    [6, 9, 2],
    [9, 14, 4],
    [2, 4, 3]
])

Qd, Td = lagrange(Q)
display(Latex("Ответ"))
print(geo(Qd))
print(geo(Td))

<IPython.core.display.Latex object>

Matrix([
[6,  9, 2],
[9, 14, 4],
[2,  4, 3]])

<IPython.core.display.Latex object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

Matrix([
[1, -3, 22/21],
[0,  1,  -4/7],
[0,  0,     1]])

<IPython.core.display.Latex object>

Matrix([
[   6, -9, 22/7],
[  -9, 14,   -4],
[22/7, -4,    3]])

<IPython.core.display.Latex object>

[6, -9, 3.1428571429; -9, 14, -4; 3.1428571429, -4, 3]
[1, -3, 1.0476190476; 0, 1, -0.5714285714; 0, 0, 1]


[6, 0, 0; 0, 1/2, 0; 0, 0, 1/3]
[1, -3/2, 8/3; 0, 1, -2; 0, 0, 1]

In [40]:
C = Matrix([
    [1, 3/2, 1/3],
    [0, 1, 2],
    [0, 0, 1]
])
Q = Matrix([
    [6, 9, 2],
    [9, 14, 4],
    [2, 4, 3]
])

C = C.inv()
Qd = C.T * Q * C
display(Qd)
print(geo(Qd))
print(geo(C))
print(geo(C.T))
print(geo(C.inv()))
print(geo(C.inv().T))
print(geo(C.T.inv()))

Matrix([
[6.0,   0,                 0],
[  0, 0.5,                 0],
[  0,   0, 0.333333333333333]])

[6.00000000000000, 0, 0; 0, 0.5000000000, 0; 0, 0, 0.3333333333]
[1.00000000000000, -1.5000000000, 2.6666666667; 0, 1.00000000000000, -2.00000000000000; 0, 0, 1.00000000000000]
[1.00000000000000, 0, 0; -1.5000000000, 1.00000000000000, 0; 2.6666666667, -2.00000000000000, 1.00000000000000]
[1.00000000000000, 1.5000000000, 0.3333333333; 0, 1.00000000000000, 2.00000000000000; 0, 0, 1.00000000000000]
[1.00000000000000, 0, 0; 1.5000000000, 1.00000000000000, 0; 0.3333333333, 2.00000000000000, 1.00000000000000]
[1.00000000000000, 0, 0; 1.5000000000, 1.00000000000000, 0; 0.3333333333, 2.00000000000000, 1.00000000000000]


[6, 0, 0; 0, 1/2, 0; 0, 0, 1/3]
[1, -1.5, 2.6666; 0, 1, -2; 0, 0, 1]
[1, 0, 0; -1.5, 1, 0; 2.6667, -2, 1]
[1, 1.5, 0.3333; 0, 1, 2; 0, 0, 1]
[1, 0, 0; 1.5, 1, 0; 0.3333, 2, 1]
[1, 0, 0; 1.5, 1, 0; 0.3333, 2, 1]

In [ ]:
import numpy as np

# Reading number of unknowns
n = int(input('Enter number of data points: '))

# Making numpy array of n & n x n size and initializing 
# to zero for storing x and y value along with differences of y
x = np.zeros((n))
y = np.zeros((n))


# Reading data points
print('Enter data for x and y: ')
for i in range(n):
    x[i] = float(input( 'x['+str(i)+']='))
    y[i] = float(input( 'y['+str(i)+']='))


# Reading interpolation point
xp = float(input('Enter interpolation point: '))

# Set interpolated value initially to zero
yp = 0

# Implementing Lagrange Interpolation
for i in range(n):
    
    p = 1
    
    for j in range(n):
        if i != j:
            p = p * (xp - x[j])/(x[i] - x[j])
    
    yp = yp + p * y[i]    

# Displaying output
print('Interpolated value at %.3f is %.3f.' % (xp, yp))